# 01 — SfM-30k feature cache

Build the frozen DINOv2 ViT-S/14-with-registers cache used by every SfM experiment. Put official SfM files in `data/raw/sfm30k/`; missing files are downloaded there automatically.

In [ ]:
import gc
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from matplotlib.ticker import PercentFormatter

from cbir.backbone import FrozenDinoV2Extractor
from cbir.cache import FeatureShardReader
from cbir.config import config_to_dict, load_project_config
from cbir.data.sfm import Sfm30kMetadata, SfmMatImageReader
from cbir.features import FeatureExtractionRunner
from cbir.plotting import SeriesData, plot_series
from cbir.utils import atomic_write_json, read_json
from cbir.workflow import build_sfm_feature_cache, ensure_sfm30k_sources

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from the project root or notebooks directory.')
os.chdir(PROJECT_ROOT)
TORCH_HOME = PROJECT_ROOT / 'data' / 'models' / 'torch_hub'
os.environ.setdefault('TORCH_HOME', str(TORCH_HOME))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'local.yaml'
cfg = load_project_config(CONFIG_PATH)
CACHE_DIR = cfg.cache.root / 'sfm30k'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / '01_sfm_feature_cache'
FIGURES_DIR = OUTPUT_DIR / 'figures'
RESULTS_PATH = OUTPUT_DIR / 'results.json'
BACKBONE_BATCH_SIZE = 8
DECODED_IMAGE_CHUNK_SIZE = 128
PILOT_IMAGES_PER_SPLIT = 500
PILOT_TEMPERATURES = (0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_ROOT)
print('SfM source:', cfg.sfm.image_mat_path)
print('Feature cache:', CACHE_DIR)
print('Torch Hub cache:', os.environ['TORCH_HOME'])
print('Backbone batch size:', BACKBONE_BATCH_SIZE)
print('Decoded-image chunk size:', DECODED_IMAGE_CHUNK_SIZE)

## Prepare SfM-30k

The compact 30k MAT file is used directly and read lazily during feature extraction.

In [ ]:
ensure_sfm30k_sources(cfg)
metadata = Sfm30kMetadata.from_official_files(
    cfg.sfm.metadata_path,
    cfg.sfm.names_clusters_path,
)
display(pd.DataFrame([{
    'images': len(metadata.images),
    'train_pairs': len(metadata.train_pairs),
    'validation_pairs': len(metadata.val_pairs),
}]))

## Pooling-temperature pilot

This reproduces the SfM-only diagnostic used to select `τₚ = 0.025`. It does not create a feature cache.

In [ ]:
pilot_records = (
    metadata.records('train')[:PILOT_IMAGES_PER_SPLIT]
    + metadata.records('val')[:PILOT_IMAGES_PER_SPLIT]
)
pilot_spec = {
    'image_ids': [record.image_id for record in pilot_records],
    'temperatures': list(PILOT_TEMPERATURES),
    'layers_zero_based': list(cfg.pooling.all_layer_indices),
    'backbone': config_to_dict(cfg.backbone),
    'preprocess': config_to_dict(cfg.preprocess),
    'pooling': config_to_dict(cfg.pooling),
    'decoded_image_chunk_size': DECODED_IMAGE_CHUNK_SIZE,
}
notebook_results = read_json(RESULTS_PATH) if RESULTS_PATH.is_file() else {}
saved_pilot = notebook_results.get('pooling_pilot', {})
if saved_pilot.get('run_spec') == pilot_spec:
    pilot_results = saved_pilot['results']
    print('Reusing saved pooling-temperature pilot.')
else:
    if any(record.split is None or record.image_locator is None for record in pilot_records):
        raise RuntimeError('The SfM pilot could not resolve every selected image.')

    pilot_extractor = FrozenDinoV2Extractor(cfg.backbone)
    pilot_runner = FeatureExtractionRunner(pilot_extractor, cfg.preprocess, cfg.pooling)
    with SfmMatImageReader(cfg.sfm.image_mat_path) as source:
        pilot = pilot_runner.pilot_pooling_temperatures(
            (
                (record.image_id, source.read(record.split, int(record.image_locator)))
                for record in pilot_records
            ),
            temperatures=PILOT_TEMPERATURES,
            layer_indices=cfg.pooling.all_layer_indices,
            backbone_batch_size=BACKBONE_BATCH_SIZE,
            image_chunk_size=DECODED_IMAGE_CHUNK_SIZE,
        )
    pilot_results = {
        'layers_zero_based': list(pilot.layer_indices),
        'mean_entropy_by_temperature': {
            str(temperature): pilot.entropy_by_temperature[temperature].mean(dim=0).tolist()
            for temperature in PILOT_TEMPERATURES
        },
        'mean_guided_mean_cosine_by_temperature': {
            str(temperature): pilot.guided_mean_cosine_by_temperature[temperature].mean(dim=0).tolist()
            for temperature in PILOT_TEMPERATURES
        },
    }
    notebook_results['pooling_pilot'] = {'run_spec': pilot_spec, 'results': pilot_results}
    atomic_write_json(RESULTS_PATH, notebook_results)
    del pilot, pilot_runner, pilot_extractor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
layer_positions = pilot_results['layers_zero_based']
entropy_series = {
    str(temperature): SeriesData(
        x=layer_positions,
        y=pilot_results['mean_entropy_by_temperature'][str(temperature)],
    )
    for temperature in PILOT_TEMPERATURES
}
entropy_figure, entropy_axis = plot_series(
    entropy_series,
    title='Mean normalized pooling entropy',
    xlabel='Transformer block index (0 = first)',
    ylabel='Entropy',
    legend_title='Pooling temperature (τₚ)',
    fig_size=(11, 6),
    save_path=FIGURES_DIR / 'pooling_entropy.png',
)
display(entropy_figure)
plt.close(entropy_figure)

similarity_series = {
    str(temperature): SeriesData(
        x=layer_positions,
        y=pilot_results['mean_guided_mean_cosine_by_temperature'][str(temperature)],
    )
    for temperature in PILOT_TEMPERATURES
}
similarity_figure, similarity_axis = plot_series(
    similarity_series,
    title='CLS-guided versus mean-patch local descriptor',
    xlabel='Transformer block index (0 = first)',
    ylabel='Mean cosine similarity',
    legend_title='Pooling temperature (τₚ)',
    fig_size=(11, 6),
    save_path=FIGURES_DIR / 'pooling_guided_mean_similarity.png',
)
similarity_axis.yaxis.set_major_formatter(PercentFormatter(1.0))
display(similarity_figure)
plt.close(similarity_figure)

## Extract or reuse frozen features

All twelve transformer blocks are cached once. Later notebooks never reopen the SfM images.

In [ ]:
cache_location = build_sfm_feature_cache(
    cfg,
    metadata=metadata,
    backbone_batch_size=BACKBONE_BATCH_SIZE,
    image_chunk_size=DECODED_IMAGE_CHUNK_SIZE,
)
reader = FeatureShardReader(cache_location.cache_dir, preload=True)
cache_summary = {
    'cache_dir': str(cache_location.cache_dir),
    'images': len(reader.image_ids),
    'shards': len(reader.manifest.shards),
    'layers': list(reader.manifest.layer_indices),
    'fingerprint': reader.manifest.fingerprint,
    'decoded_image_chunk_size': DECODED_IMAGE_CHUNK_SIZE,
    'config': config_to_dict(cfg),
}
notebook_results = read_json(RESULTS_PATH) if RESULTS_PATH.is_file() else {}
notebook_results['feature_cache'] = cache_summary
atomic_write_json(RESULTS_PATH, notebook_results)
display(pd.DataFrame([{key: value for key, value in cache_summary.items() if key != 'config'}]))